In [ ]:
# Library CRF untuk Viterbi decoding
%pip install -q pytorch-crf==0.7.2

print("✓ Cell 1 berhasil: pytorch-crf sudah terpasang.")

In [ ]:
import json
import re
from pathlib import Path

import torch
import torch.nn as nn

from google.colab import drive
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)
from torchcrf import CRF

drive.mount("/content/drive")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/BiLSTM-CRF"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "finetuned"
    / "best_finetuned_model.pt"
)

CHAR2ID_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "char2id.json"
)

LABEL2ID_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "label2id.json"
)

for file_path in [
    MODEL_PATH,
    CHAR2ID_PATH,
    LABEL2ID_PATH,
]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"File tidak ditemukan: {file_path}"
        )

with open(
    CHAR2ID_PATH,
    encoding="utf-8",
) as file:
    char2id = json.load(file)

with open(
    LABEL2ID_PATH,
    encoding="utf-8",
) as file:
    label2id = json.load(file)

id2label = {
    index: label
    for label, index in label2id.items()
}

PAD_CHAR_ID = char2id["<PAD>"]
UNK_CHAR_ID = char2id["<UNK>"]

EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.30
MAX_LEN = 300

print("Device          :", DEVICE)
print("Vocabulary size :", len(char2id))
print("Number of labels:", len(label2id))
print("Model path      :", MODEL_PATH)
print("✓ Cell 2 berhasil dijalankan.")

In [ ]:
class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        pad_char_id,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_char_id,
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        self.output_dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim,
            num_labels,
        )

        self.crf = CRF(
            num_tags=num_labels,
            batch_first=True,
        )

    def compute_emissions(
        self,
        char_ids,
        lengths,
    ):
        embedded = self.embedding(char_ids)
        embedded = self.embedding_dropout(embedded)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_output, _ = self.bilstm(packed)

        encoded, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=char_ids.size(1),
        )

        encoded = self.output_dropout(encoded)

        return self.classifier(encoded)

    def decode(
        self,
        char_ids,
        mask,
        lengths,
    ):
        emissions = self.compute_emissions(
            char_ids,
            lengths,
        )

        return self.crf.decode(
            emissions,
            mask=mask,
        )


model = BiLSTMCRF(
    vocab_size=len(char2id),
    num_labels=len(label2id),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_char_id=PAD_CHAR_ID,
).to(DEVICE)


checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)

assert checkpoint["char2id"] == char2id
assert checkpoint["label2id"] == label2id

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

assert not load_result.missing_keys
assert not load_result.unexpected_keys

model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)

print("Model           : BiLSTM-CRF Fine-tuned")
print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Validation loss:",
    round(
        float(checkpoint["validation_loss"]),
        4,
    ),
)
print("Status          : siap digunakan")
print("✓ Cell 3 berhasil dijalankan.")

In [ ]:
HARAKAT = set("ًٌٍَُِّْ")


def remove_diacritics(text):
    return "".join(
        char
        for char in text
        if char not in HARAKAT
    )


def predict_diacritics(text):
    # Menghapus harakat jika pengguna memasukkan
    # teks yang sudah berharakat
    clean_text = remove_diacritics(text)

    # Merapikan spasi dan baris baru
    clean_text = re.sub(
        r"\s+",
        " ",
        clean_text,
    ).strip()

    if not clean_text:
        raise ValueError(
            "Teks input tidak boleh kosong."
        )

    if len(clean_text) > MAX_LEN:
        raise ValueError(
            f"Panjang input {len(clean_text)} karakter. "
            f"Maksimum {MAX_LEN} karakter."
        )

    unknown_characters = sorted({
        char
        for char in clean_text
        if char not in char2id
    })

    char_ids = [
        char2id.get(char, UNK_CHAR_ID)
        for char in clean_text
    ]

    char_tensor = torch.tensor(
        [char_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    mask = torch.ones(
        char_tensor.shape,
        dtype=torch.bool,
        device=DEVICE,
    )

    lengths = torch.tensor(
        [len(char_ids)],
        dtype=torch.long,
    )

    with torch.inference_mode():
        predicted_ids = model.decode(
            char_tensor,
            mask,
            lengths,
        )[0]

    output = []

    for char, label_id in zip(
        clean_text,
        predicted_ids,
    ):
        label = id2label[label_id]

        output.append(char)

        if label != "O":
            output.extend(
                label.split("|")
            )

    predicted_text = "".join(output)

    return {
        "input": clean_text,
        "prediction": predicted_text,
        "length": len(clean_text),
        "unknown_characters": (
            unknown_characters
        ),
    }


# Input langsung dari pengguna
TEST_TEXT = input(
    "Masukkan teks Arab gundul: "
)

result = predict_diacritics(TEST_TEXT)

print("\nTeks Arab gundul:")
print(result["input"])

print("\nHasil diakritisasi:")
print(result["prediction"])

print("\nPanjang karakter :", result["length"])
print(
    "Karakter unknown :",
    result["unknown_characters"],
)

print("✓ Pengujian model selesai.")

In [ ]:
import unicodedata


def normalize_text(text):
    text = unicodedata.normalize("NFC", text)

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def split_long_text(text, max_length=300):
    words = text.split()
    chunks = []
    current_chunk = []

    for word in words:
        candidate = " ".join(
            current_chunk + [word]
        )

        if len(candidate) <= max_length:
            current_chunk.append(word)
        else:
            if current_chunk:
                chunks.append(
                    " ".join(current_chunk)
                )

            if len(word) > max_length:
                raise ValueError(
                    "Terdapat satu kata melebihi "
                    f"{max_length} karakter."
                )

            current_chunk = [word]

    if current_chunk:
        chunks.append(
            " ".join(current_chunk)
        )

    return chunks


def predict_long_text(text):
    clean_text = normalize_text(
        remove_diacritics(text)
    )

    chunks = split_long_text(
        clean_text,
        MAX_LEN,
    )

    predictions = []
    unknown_characters = set()

    for chunk in chunks:
        chunk_result = predict_diacritics(chunk)

        predictions.append(
            chunk_result["prediction"]
        )

        unknown_characters.update(
            chunk_result["unknown_characters"]
        )

    return {
        "input": clean_text,
        "prediction": " ".join(predictions),
        "chunks": len(chunks),
        "unknown_characters": sorted(
            unknown_characters
        ),
    }


def extract_char_marks(text):
    text = normalize_text(text)

    chars = []
    marks_per_char = []
    index = 0

    while index < len(text):
        char = text[index]

        if char in HARAKAT:
            raise ValueError(
                "Ditemukan harakat tanpa "
                "karakter dasar."
            )

        chars.append(char)

        marks = set()
        next_index = index + 1

        while (
            next_index < len(text)
            and text[next_index] in HARAKAT
        ):
            marks.add(text[next_index])
            next_index += 1

        marks_per_char.append(
            frozenset(marks)
        )

        index = next_index

    return chars, marks_per_char


def is_arabic_letter(char):
    return (
        len(char) == 1
        and "ARABIC" in unicodedata.name(
            char,
            "",
        )
        and unicodedata.category(
            char
        ).startswith("L")
    )


def evaluate_single_reference(reference_text):
    reference_text = normalize_text(
        reference_text
    )

    result = predict_long_text(
        reference_text
    )

    reference_chars, reference_marks = (
        extract_char_marks(reference_text)
    )

    predicted_chars, predicted_marks = (
        extract_char_marks(
            result["prediction"]
        )
    )

    assert reference_chars == predicted_chars
    assert len(reference_marks) == len(
        predicted_marks
    )

    total_characters = len(reference_chars)

    correct_characters = sum(
        reference == prediction
        for reference, prediction in zip(
            reference_marks,
            predicted_marks,
        )
    )

    arabic_positions = [
        index
        for index, char in enumerate(
            reference_chars
        )
        if is_arabic_letter(char)
    ]

    correct_arabic = sum(
        reference_marks[index]
        == predicted_marks[index]
        for index in arabic_positions
    )

    accuracy = (
        correct_characters
        / total_characters
        * 100
    )

    arabic_accuracy = (
        correct_arabic
        / len(arabic_positions)
        * 100
    )

    der = 100 - arabic_accuracy

    reference_words = reference_text.split()
    predicted_words = result[
        "prediction"
    ].split()
    base_words = result["input"].split()

    assert (
        len(reference_words)
        == len(predicted_words)
        == len(base_words)
    )

    total_words = 0
    incorrect_words = 0

    for base, reference, prediction in zip(
        base_words,
        reference_words,
        predicted_words,
    ):
        if not any(
            is_arabic_letter(char)
            for char in base
        ):
            continue

        total_words += 1

        if (
            unicodedata.normalize(
                "NFD",
                reference,
            )
            != unicodedata.normalize(
                "NFD",
                prediction,
            )
        ):
            incorrect_words += 1

    wer = (
        incorrect_words
        / total_words
        * 100
    )

    return {
        "input": result["input"],
        "reference": reference_text,
        "prediction": result["prediction"],
        "chunks": result["chunks"],
        "unknown_characters": (
            result["unknown_characters"]
        ),
        "accuracy": accuracy,
        "arabic_accuracy": arabic_accuracy,
        "der": der,
        "wer": wer,
        "correct_characters": (
            correct_characters
        ),
        "total_characters": total_characters,
        "incorrect_words": incorrect_words,
        "total_words": total_words,
    }


REFERENCE_TEXT = input(
    "Masukkan teks referensi berharakat: "
)

evaluation = evaluate_single_reference(
    REFERENCE_TEXT
)

print("\nTeks Arab gundul:")
print(evaluation["input"])

print("\nReferensi:")
print(evaluation["reference"])

print("\nPrediksi model:")
print(evaluation["prediction"])

print("\nHASIL PENGUJIAN")
print(
    "Accuracy keseluruhan :",
    f'{evaluation["accuracy"]:.4f}%',
)
print(
    "Ketepatan huruf Arab :",
    f'{evaluation["arabic_accuracy"]:.4f}%',
)
print(
    "DER                  :",
    f'{evaluation["der"]:.4f}%',
)
print(
    "WER                  :",
    f'{evaluation["wer"]:.4f}%',
)
print(
    "Karakter benar       :",
    evaluation["correct_characters"],
    "dari",
    evaluation["total_characters"],
)
print(
    "Kata salah           :",
    evaluation["incorrect_words"],
    "dari",
    evaluation["total_words"],
)
print(
    "Jumlah potongan      :",
    evaluation["chunks"],
)
print(
    "Karakter unknown     :",
    evaluation["unknown_characters"],
)